# Example — the test concept graph

How a consumer can use the cdisc-for-all graphs for study design: a SPARQL CONSTRUCT query returns a subgraph, every triple as it is in the graphs. Nothing is added and nothing is written back to the repository.

`test_concept_graph.rq` — what COSMoS and NCIt say about a test, without the SDTM implementation. The start is the C-code of the test term (the Code column of SDTM CT; example C105585, GLUC / Glucose): its NCIt concept leads to the COSMoS BC with the same concept (Glucose Measurement); the result is the CT terms with that concept, the BC side of COSMoS with NCIt, the SDTM CT terms that Dataset Specialization rows bind to the BC's DECs, and the Specializations themselves. It shows what is there: only the test and its concept if no COSMoS BC has the concept; the BC without DECs if it has none. The boundary follows COSMoS's own split into a BC model and an SDTM model.

`test_concept.py` runs the query for one or more C-codes and writes per code the result graph and a page to read it; this notebook calls the same functions. From a terminal: `python examples/test_concept.py C105585 C64433`.

The query runs in Oxigraph (pyoxigraph); rdflib's SPARQL engine takes too long over the 1.5 million triples. The result and the page go to `examples/output/` (not in git).

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
from examples.test_concept import load_store, test_concept, write

OUT = REPO_ROOT / 'examples' / 'output'
RDF_TYPE = 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type'
CODE = 'C105585'

store = load_store()
print(len(store), 'triples')

## The result for one test

Change `CODE` above for another test.

In [ ]:
result = test_concept(store, CODE)
nt, page, title, level = write(result, CODE, OUT)
print(title, len(result), 'triples,', level)
print('written', nt.relative_to(REPO_ROOT).as_posix())
print('written', page.relative_to(REPO_ROOT).as_posix(), '(open it in a browser)')
types = Counter(q.object.value for q in result.quads_for_pattern(None, None, None) if q.predicate.value == RDF_TYPE)
print(pd.Series(types, name='nodes').sort_index().to_string())

## The DECs and their bound CT terms, as a table

In [ ]:
select = """
PREFIX bc: <https://kerfors.github.io/cdisc-for-all/def/cosmos/cdisc_biomedical_concepts_latest#>
PREFIX ct: <https://kerfors.github.io/cdisc-for-all/def/sdtm-ct/SDTM%20Terminology#>
SELECT ?dec_id ?dec_label ?data_type ?example_set ?codelist
       (GROUP_CONCAT(DISTINCT ?term; separator=";") AS ?bound_terms) WHERE {
    ?dec bc:dec_id ?dec_id ; bc:dec_label ?dec_label .
    OPTIONAL { ?dec bc:data_type ?data_type }
    OPTIONAL { ?dec bc:example_set ?example_set }
    OPTIONAL {
        ?row <https://kerfors.github.io/cdisc-for-all/link/cosmos/cdisc_sdtm_dataset_specializations_latest#dec_id> ?dec ;
             <https://kerfors.github.io/cdisc-for-all/link/cosmos/cdisc_sdtm_dataset_specializations_latest#assigned_term> |
             <https://kerfors.github.io/cdisc-for-all/rule/cosmos/cdisc_sdtm_dataset_specializations_latest#value_list/sdtm-ct/SDTM%20Terminology> ?t .
        ?t ct:CDISC%20Submission%20Value ?term ; ct:Codelist%20Name ?codelist .
    }
} GROUP BY ?dec_id ?dec_label ?data_type ?example_set ?codelist ORDER BY ?dec_label ?codelist"""
rows = [[t.value if t is not None else '' for t in s] for s in result.query(select)]
pd.DataFrame(rows, columns=[v.value for v in result.query(select).variables])